In [ ]:
import pandas as pd
import plotly.graph_objects as go
import webbrowser

In [ ]:
df_cn = pd.read_excel("C:\\Users\\38670\\Documents\\Documents\\总体和青年失业率.xlsx")
df_cn.columns = ['Year', 'CN_Total_Unemployment', 'CN_Youth_Unemployment']

In [ ]:
df_us_total = pd.read_csv("C:\\Users\\38670\\Documents\\Documents\\整体失业率.csv")
df_us_youth = pd.read_csv("C:\\Users\\38670\\Documents\\Documents\\Unemployment Rate (Youth, ages 16–24).csv")

In [ ]:
df_us_total_clean = df_us_total[['Year', 'Value']].copy()
df_us_total_clean.columns = ['Year', 'US_Total_Unemployment']

In [ ]:
df_us_youth['Year'] = pd.to_datetime(df_us_youth['observation_date']).dt.year
df_us_youth_clean = df_us_youth[['Year', 'LNS14024887']]
df_us_youth_clean.columns = ['Year', 'US_Youth_Unemployment']

合并数据

In [ ]:
df_us = pd.merge(df_us_total_clean, df_us_youth_clean, on='Year', how='inner')
df_all = pd.merge(df_cn, df_us, on='Year', how='inner')

In [ ]:
for col in ['CN_Total_Unemployment', 'CN_Youth_Unemployment',
            'US_Total_Unemployment', 'US_Youth_Unemployment']:
    if df_all[col].max() < 1:
        df_all[col] *= 100

In [ ]:
fig = go.Figure()

In [ ]:
fig.add_trace(go.Scatter(x=df_all['Year'], y=df_all['CN_Total_Unemployment'],
                         mode='lines+markers', name='China - Total', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=df_all['Year'], y=df_all['CN_Youth_Unemployment'],
                         mode='lines+markers', name='China - Youth (16–24)', line=dict(color='green')))
fig.add_trace(go.Scatter(x=df_all['Year'], y=df_all['US_Total_Unemployment'],
                         mode='lines+markers', name='US - Total', line=dict(color='orange', dash='dot')))
fig.add_trace(go.Scatter(x=df_all['Year'], y=df_all['US_Youth_Unemployment'],
                         mode='lines+markers', name='US - Youth (16–24)', line=dict(color='red', dash='dot')))

In [ ]:
#标注危机年份
for year in [2008, 2020]:
    fig.add_vline(x=year, line=dict(color='gray', dash='dot'))
    fig.add_annotation(x=year, y=20,  # ✅ 这里修复了原来炸掉的 y=max(df_all.max())
                       text=f"{year} crisis", showarrow=False,
                       font=dict(color="gray", size=11))

In [ ]:
fig.update_layout(
    title="Unemployment Rate Trends: China and US (2000–2023)",
    xaxis_title="Year",
    yaxis_title="Unemployment Rate (%)",
    template="plotly_white",
    legend_title="Country & Age Group",
    yaxis=dict(range=[0, 25])  # 限制Y轴范围，避免爆炸
)

In [ ]:
output_path = "unemployment_trend_plotly_fixed.html"
fig.write_html(output_path)
webbrowser.open(output_path)